# Eksplorasi AutoModelForMaskedLM

**Task**: Masked Language Modeling (Menebak kata yang disembunyikan/di-mask)
**Cara Kerja**: Model menerima satu kalimat utuh di mana salah satu atau beberapa token diganti dengan token spesial `[MASK]`. Model bertugas membaca konteks kiri dan kanan dari token yang disembunyikan itu, lalu menebak apa token sebenarnya yang tertutup tersebut.
**Model Populer**: BERT, RoBERTa, ALBERT, DistilBERT (Varian Encoder murni).
**Dataset**: Tidak menggunakan dataset spesifik karena proses Masked LM bersifat "Self-Supervised". Cukup meload dataset teks panjang apa saja (misal wiki teks, buku, cnn news), dan menutupi sebagian kata secara acak di setiap iterasi latih.

In [1]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline
import torch

## 1. Load Tokenizer & Model
Kita akan menggunakan bapak dari segala model Masked LM, yaitu `bert-base-uncased`. BERT adalah pelopor utama inovasi ini.

In [2]:
model_checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Menggunakan AutoModelForMaskedLM!
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
print(model)

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

## 2. Inferensi Manual
Pada Masked LM, kita sediakan text, lalu tambahkan token target khusus sesuai dengan arsitektur model. Untuk model keluarga BERT, token yang digunakan untuk kata misteri adalah string `[MASK]`.

In [15]:
text = f"Paris is the {tokenizer.mask_token} of France."
print("Kalimat dengan token mask:", text)

inputs = tokenizer(text, return_tensors="pt")

print(f'inputs : {inputs}')

# Mencari di mana posisi index array [MASK] secara berurutan dalam kalimat tokenisasi
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[-1]

print(f"mask token {mask_token_index}")
with torch.no_grad():
    outputs = model(**inputs)
    print(outputs.logits.shape)  
# Terdapat banyak token yang diprediksi model, namun kita hanya butuh nilai list probabilitas prediksi milik token yang ada di lokasi `mask_token_index`
mask_token_logits = outputs.logits[0, mask_token_index, :]
print(mask_token_logits.shape)
# # Cari top 5 kandidat tebakannya:
top_5_tokens = torch.topk(mask_token_logits, 5, dim=1)

top_5_tokens = torch.topk(mask_token_logits, 5, dim=1).indices[0].tolist()

print("\n--- Prediksi Top 5 Kandidat (Inferensi Manual) ---")
for token_id in top_5_tokens:
    token_word = tokenizer.decode([token_id])
    predicted_sentence = text.replace(tokenizer.mask_token, token_word)
    print(f"Tebakan kata: {token_word:15} -> {predicted_sentence}")

Kalimat dengan token mask: Paris is the [MASK] of France.
inputs : {'input_ids': tensor([[ 101, 3000, 2003, 1996,  103, 1997, 2605, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}
mask token tensor([4])
torch.Size([1, 9, 30522])
torch.Size([1, 30522])

--- Prediksi Top 5 Kandidat (Inferensi Manual) ---
Tebakan kata: capital         -> Paris is the capital of France.
Tebakan kata: heart           -> Paris is the heart of France.
Tebakan kata: center          -> Paris is the center of France.
Tebakan kata: centre          -> Paris is the centre of France.
Tebakan kata: city            -> Paris is the city of France.


In [27]:
tokenizer.mask_token_id

103

## 3. Persiapan Data untuk Training Manual (PyTorch)
Untuk melatih Masked LM, model ini bertumpu pada **Self-Supervised Learning**. Kita cukup menyiapkan naskah teks bebas (tanpa label manual).

Sebagai ganti label manual, Hugging Face memiliki utilitas ajaib bernama `DataCollatorForLanguageModeling`. Kelas ini bertugas untuk menyuntikkan token `[MASK]` secara otomatis (secara deafult peluang disembunyikan ~15% kata per kalimat) *on-the-fly* setiap kali data masukan dimasukan ke dalam Batch size. Sekaligus melengkapi parameter *labels* bernilai tensor kata aslinya.

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling

# 1. Simulasikan corpus teks sederhana dari dataset wikitext
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
dataset = dataset.filter(lambda x: len(x['text'].strip()) > 50)
train_sample = dataset.select(range(50)) # Pakai porsi super kecil

# 2. Setup fungsi Tokenisasi
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, max_length=128)

tokenized_train = train_sample.map(tokenize_function, batched=True, remove_columns=['text'])

# 3. Data Collator khusus Masked LM (wajib set parameter mlm=True)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=True, mlm_probability=0.15
)

# 4. DataLoader yang tersambung dengan Data Collator buatan
train_dataloader = DataLoader(
    tokenized_train, shuffle=True, batch_size=4, collate_fn=data_collator
)

print(f"Total Batch (Iterasi): {len(train_dataloader)}")
print("Persiapan dataset Self-Supervised [MASK] selesai!")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Total Batch (Iterasi): 13
Persiapan dataset Self-Supervised [MASK] selesai!


In [37]:
for data in train_dataloader:
    print(data['input_ids'].shape)

torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([4, 128])
torch.Size([2, 128])


## 4. Proses PyTorch Training Loop
Loop ini bersifat absolut (Forward, Backward, Optimizer, Step). Ketika forward pass, AutoModelForMaskedLM akan mengeksekusi logika internal untuk hanya menghitung `Loss` pada kata/token yang di-[MASK] saja (token dengan nilai `-100` pada labels). Token yang tidak disamakan akan diabaikan pada saat perhitungan galat error.

In [38]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 1
print("==== Memulai Training MLM ====")
model.train()
for epoch in range(epochs):
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        # Ekstrak data dari dictionary ke device hardware
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Forward pass (Loss langsung auto terhitung untuk token [MASK] saja)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        # Backpropagation
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 5 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss Saat Ini : {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f">> Rata-rata Train Loss Epoch {epoch+1}: {avg_train_loss:.4f}\n")

==== Memulai Training MLM ====
Epoch 1 | Step 0 | Loss Saat Ini : 2.0516
Epoch 1 | Step 5 | Loss Saat Ini : 1.8050
Epoch 1 | Step 10 | Loss Saat Ini : 2.7755
>> Rata-rata Train Loss Epoch 1: 2.2577



## 5. Evaluasi Kinerja (Metrik Evaluasi Masked LM)

Karena Masked LM bukanlah klasifikasi Label biasa (bukan Prediksi Sentimen), untuk menilai keberhasilannya kita bisa menggunakan beberapa macam metrik:
1. **Cross-Entropy Loss (Validation Loss):** Metrik validasi termudah. Jika bobot Validation Loss semakin kecil, ia memprediksi pola kemunculan token di korpus teks baru tersebut dengan performa yang lebih akurat.
2. **Pseudo-Perplexity (PPL):** Secara literatur NLP, _Perplexity_ ($\exp(\text{loss})$) umumnya diciptakan untuk ranah AutoRegulatory/Causal LM (misal: menebak _Next Token_ / GPT-2). Namun untuk Masked Language Model layaknya keluarga BERT, komputasinya disebut *Pseudo-Perplexity* karena ia adalah "tebak menyilang kiri-kanan secara bersamaan", tapi bisa tetap dihitung asalkan validation loss-nya stabil. 
3. **Accuracy (Akruasi Kata Exact):** Ini mengharuskan looping manual—Anda memaksanya menghitung per indeks di mana terjadi *MASK*, apabila kata tebakannya berstatus `Prediksi Top-1 == Target`, maka akurasi di-increment ke atas. Biasanya dilaporkan sebagai metrik akurasi pre-train. 

Kita akan mensimulasikan penggunaan Loss vs Pseudo-Perplexity di kode Evaluasi PyTorch sederhana ini:

In [39]:
import math

# Siapkan data evaluasi validation/test dengan teknik tokenisasi dan collator (yang merandom array MASK) sama.
val_sample = dataset.select(range(50, 70))
tokenized_val = val_sample.map(tokenize_function, batched=True, remove_columns=['text'])
val_dataloader = DataLoader(tokenized_val, batch_size=4, collate_fn=data_collator)

model.eval()
total_val_loss = 0
with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_val_loss += outputs.loss.item()

# 1. Menghitung Evaluasi Rata-rata Validation Loss (Cross Entropy)
avg_val_loss = total_val_loss / len(val_dataloader)

# 2. Menghitung Pseudo-Perplexity beradasarkan perhitungan rata-rata validation Loss
pseudo_perplexity = math.exp(avg_val_loss)

print("==== Evaluasi Performa Model Masked LM Selesai ====")
print(f"Validation Loss     : {avg_val_loss:.4f}")
print(f"Pseudo-Perplexity   : {pseudo_perplexity:.4f}")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

==== Evaluasi Performa Model Masked LM Selesai ====
Validation Loss     : 1.9304
Pseudo-Perplexity   : 6.8922


## 6. Inferensi Sederhana Menggunakan Arsitektur Di-Train (Post-Train)
Kita tes lagi dengan mode model evaluation yang sama seperti kode PyTorch manual sebelumnya untuk memastikan memori hasil Fine-tuning di-*resume*.

In [40]:
test_text = f"The capital city of France is {tokenizer.mask_token}."
inputs = tokenizer(test_text, return_tensors="pt").to(device) # Kirim ke GPU jika tersedia

# Cari dimana indeks `[MASK]` berada
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    
mask_token_logits = outputs.logits[0, mask_token_index, :]
# Ambil TOP 3 hasil Prediksi logit tebakan
top_3_tokens = torch.topk(mask_token_logits, 3, dim=1).indices[0].tolist()

print("Kalimat tes (Bocor Teks):", test_text)
print("--- Prediksi Top 3 Kandidat Hasil Finetuning PyTorch ---")
for token_id in top_3_tokens:
    token_word = tokenizer.decode([token_id])
    predicted_sentence = test_text.replace(tokenizer.mask_token, token_word)
    print(f"Tebakan kata: {token_word:15} -> {predicted_sentence}")

Kalimat tes (Bocor Teks): The capital city of France is [MASK].
--- Prediksi Top 3 Kandidat Hasil Finetuning PyTorch ---
Tebakan kata: paris           -> The capital city of France is paris.
Tebakan kata: lille           -> The capital city of France is lille.
Tebakan kata: lyon            -> The capital city of France is lyon.
